# A Hybrid Deep Reinforcement Learning and LLM Framework for Real-Time Distillation Optimization
#### AI Agents for XAI and process improvement in CDU + NSU + VDU

This notebook builds an AI Agent system using **Google Gemini** models for intelligent analysis of the
three-column distillation system: **Atmospheric Distillation Unit (ADU)**, **Naphtha Stabilizer Unit (NSU)**,
and **Vacuum Distillation Unit (VDU)** — producing **13 product streams** optimized by a 16-dim RL agent.

## Agent Personas

| Persona | Role | Focus Areas |
|---------|------|-------------|
| **Process Engineer** | Full 3-column system analysis | Column performance, 13-product yields, D95% specs, profit optimization |
| **Daily Report Developer** | Automated report generation | Daily summaries, KPIs, cross-column yield trends |
| **Corrosion Expert** | ADU overhead corrosion analysis | Dew point corrosion, NH₄Cl salt deposition, chemical treatment |

---


## 1. Setup & Configuration

In [48]:
import os
import json
import sys
from datetime import datetime, timedelta
from pathlib import Path
from typing import Optional

from google import genai
from google.genai import types
import pandas as pd
import numpy as np


# Add project root to path
PROJECT_ROOT = Path(os.getcwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")


Project root: d:\github\Distillation-column-agent


### Configuration

In [49]:
import requests as _requests  # used by OpenRouter fallback

# ── Primary: Google Gemini ────────────────────────────────────────────────
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
print(f"GEMINI_API_KEY : {'set' if GEMINI_API_KEY else ' not set'}")

if GEMINI_API_KEY:
    client = genai.Client(api_key=GEMINI_API_KEY)
    print("Primary LLM    : Gemini (google-genai SDK)")
else:
    client = None
    print("  Set GEMINI_API_KEY env var or paste it here to use Gemini.")

# ── Fallback: Nvidia Nemotron via OpenRouter ──────────────────────────────
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "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")
print(OPENROUTER_API_KEY)
OPENROUTER_MODEL   = "nvidia/nemotron-3-nano-30b-a3b:free"
OPENROUTER_BASE    = "https://openrouter.ai/api/v1/chat/completions"

print(f"OPENROUTER_KEY : {'set' if OPENROUTER_API_KEY else 'not set'}")

if not GEMINI_API_KEY and not OPENROUTER_API_KEY:
    print("\n No LLM API key configured — agents will run in offline/demo mode.")
elif not GEMINI_API_KEY and OPENROUTER_API_KEY:
    print(f"Fallback LLM   : {OPENROUTER_MODEL}")


def _openrouter_chat(system_prompt: str, user_message: str,
                     model: str = OPENROUTER_MODEL) -> str:
    """Call OpenRouter (OpenAI-compatible REST) and return response text."""
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://github.com/Distillation-column-agent",
        "X-Title": "CDU Optimizer AI Agent",
    }
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message},
        ],
        "stream": False,
    }
    resp = _requests.post(OPENROUTER_BASE, headers=headers, json=payload, timeout=90)
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"]


GEMINI_API_KEY : set
Primary LLM    : Gemini (google-genai SDK)
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
OPENROUTER_KEY : set


## 2. System Data Loader

Load simulation data, training metrics, and prices from the project.

In [50]:
def load_prices(scenario="default"):
    """Load product prices from the data file."""
    prices_path = PROJECT_ROOT / "backend" / "data" / "prices.json"
    if prices_path.exists():
        with open(prices_path) as f:
            data = json.load(f)
        key = f"prices_{scenario}"
        if key in data:
            return data[key].get("prices", {})
    # Fallback: 13 products across ADU (5) + NSU (3) + VDU (5) + feed cost
    return {
        # ADU products
        "Uncondensed_Gas": 0.30,   # refinery fuel gas
        "Heavy_Naphtha":   0.60,   # reformer feed
        "SKO":             0.75,   # jet fuel / kerosene
        "Light_Gas_Oil":   0.70,   # light diesel
        "Heavy_Gas_Oil":   0.70,   # heavy diesel / gasoil
        # NSU products
        "StabOffGas":      0.30,   # stabilizer off-gas
        "LPG":             0.65,   # liquefied petroleum gas
        "SRN":             0.75,   # straight-run naphtha
        # VDU products
        "Offgas":          0.30,   # VDU overhead gas
        "Vacuum_Diesel":   0.70,   # VDU diesel
        "Vacuum_Gas_Oil":  0.50,   # FCC / hydrocracker feed
        "Hotwell_Oil":     0.50,   # slop / recovered oil
        "Vac_residue":     0.35,   # bitumen / fuel oil
        # Feed
        "Feed_Crude":      0.40,   # crude oil cost $/kg
    }


def load_latest_training_metrics():
    """Load the latest training checkpoint metrics."""
    cp_dir = PROJECT_ROOT / "checkpoints"
    metrics_files = sorted(cp_dir.glob("*_metrics.json"), reverse=True)
    if metrics_files:
        with open(metrics_files[0]) as f:
            return json.load(f)
    return None


def get_mock_column_state():
    """
    Representative operating snapshot for all three columns:
      ADU  — Atmospheric Distillation Unit   (5 products)
      NSU  — Naphtha Stabilizer Unit         (3 products)
      VDU  — Vacuum Distillation Unit        (5 products)
    """
    return {
        # ── ADU product flows (kg/h) ──────────────────────────────────────
        "flow_Uncondensed_Gas": 25.0,
        "flow_Heavy_Naphtha":   106.0,
        "flow_SKO":             43.0,
        "flow_Light_Gas_Oil":   51.0,
        "flow_Heavy_Gas_Oil":   69.0,
        # ── NSU product flows (kg/h) ──────────────────────────────────────
        "flow_StabOffGas":      18.0,
        "flow_LPG":             32.0,
        "flow_SRN":             56.0,
        # ── VDU product flows (kg/h) ──────────────────────────────────────
        "flow_Offgas":          12.0,
        "flow_Vacuum_Diesel":   38.0,
        "flow_Vacuum_Gas_Oil":  72.0,
        "flow_Hotwell_Oil":     14.0,
        "flow_Vac_residue":     95.0,

        # ── ADU product temperatures (°C) ─────────────────────────────────
        "temp_Uncondensed_Gas": 50.0,
        "temp_Heavy_Naphtha":   155.0,
        "temp_SKO":             220.0,
        "temp_Light_Gas_Oil":   280.0,
        "temp_Heavy_Gas_Oil":   340.0,
        # ── NSU product temperatures (°C) ─────────────────────────────────
        "temp_StabOffGas":      45.0,
        "temp_LPG":             50.0,
        "temp_SRN":             90.0,
        # ── VDU product temperatures (°C) ─────────────────────────────────
        "temp_Offgas":          65.0,
        "temp_Vacuum_Diesel":   250.0,
        "temp_Vacuum_Gas_Oil":  350.0,
        "temp_Hotwell_Oil":     380.0,
        "temp_Vac_residue":     410.0,

        # ── ADU column conditions ──────────────────────────────────────────
        "top_temperature":      50.0,     # °C
        "bottom_temperature":   340.0,    # °C
        "feed_temperature":     365.0,    # °C
        "feed_flow_rate":       4736.0,   # kg/h
        "top_pressure":         101.0,    # kPa (atmospheric)
        "bottom_pressure":      116.0,    # kPa
        "condenser_duty":       42000.0,  # kW
        "reboiler_duty":        46000.0,  # kW
        "heater_duty":          8500.0,   # kW (atmospheric furnace)
        "reflux_ratio":         5.0,      # dimensionless

        # ── NSU column conditions ──────────────────────────────────────────
        "nsu_top_temperature":  45.0,     # °C
        "nsu_bottom_temperature": 155.0,  # °C
        "nsu_top_pressure":     800.0,    # kPa (pressurised)
        "nsu_reflux_ratio":     4.0,
        "nsu_condenser_duty":   8500.0,   # kW
        "nsu_reboiler_duty":    9200.0,   # kW

        # ── VDU column conditions ──────────────────────────────────────────
        "vac_top_pressure":     8.0,      # kPa (deep vacuum)
        "vac_bottom_pressure":  15.0,     # kPa
        "vac_condenser_duty":   12000.0,  # kW
        "vac_reboiler_duty":    18000.0,  # kW
        "vac_bottom_temperature": 410.0,  # °C
        "vac_furnace_duty":     4200.0,   # kW
        "vac_reflux_ratio":     3.5,

        # ── ADU overhead corrosion monitoring ─────────────────────────────
        "overhead_temperature":     50.0,   # °C
        "overhead_pressure":        101.0,  # kPa
        "overhead_water_content":   0.02,   # mass fraction
        "overhead_hcl_ppm":         5.0,
        "overhead_h2s_ppm":         15.0,
        "overhead_nh3_ppm":         8.0,
    }


# Load data
prices = load_prices()
training_data = load_latest_training_metrics()
column_state = get_mock_column_state()

print("Prices loaded:", list(prices.keys()))
print(f"  → {len([k for k in prices if k != 'Feed_Crude'])} products + feed cost")
print("Training data:", "available" if training_data else "none")
print("Column state keys:", len(column_state))


Prices loaded: ['Uncondensed_Gas', 'USN', 'HN', 'SKO', 'LD', 'HD', 'Vac_Diesel', 'VGO', 'Slop_Cut', 'Vac_Residue', 'Feed_Crude']
  → 10 products + feed cost
Training data: available
Column state keys: 55


---

## 3. Agent Base Class

A reusable base for all AI agent personas with Gemini integration.

In [51]:
class BaseAgent:
    """
    Base AI Agent with Gemini (primary) and Nvidia Nemotron via OpenRouter (fallback).

    Priority:
      1. Google Gemini  — if GEMINI_API_KEY is set
      2. OpenRouter     — if OPENROUTER_API_KEY is set (uses nvidia/nemotron-3-nano-30b-a3b:free)
      3. Offline mode   — if neither key is available
    """

    def __init__(self, name: str, system_prompt: str, model_name: str = "gemini-2.0-flash"):
        self.name = name
        self.system_prompt = system_prompt
        self.model_name = model_name
        self.conversation_history = []

        # Set up Gemini chat session if key is available
        if client:
            self._config = types.GenerateContentConfig(
                system_instruction=system_prompt,
                temperature=0.3,
                max_output_tokens=8192,
            )
            self.chat = client.chats.create(model=model_name, config=self._config)
        else:
            self._config = None
            self.chat = None

    def ask(self, question: str, context: Optional[dict] = None) -> str:
        """Send a question to the agent and get a response."""
        # Build context string
        ctx_parts = []
        if context:
            for key, value in context.items():
                if isinstance(value, dict):
                    ctx_parts.append(f"**{key}:**\n```json\n{json.dumps(value, indent=2, default=str)}\n```")
                else:
                    ctx_parts.append(f"**{key}:** {value}")

        context_str = "\n\n".join(ctx_parts) if ctx_parts else "No additional context provided."

        full_prompt = (
            f"**System Context:**\n{context_str}\n\n"
            f"**Question:** {question}\n\n"
            "Think step-by-step through the relevant principles, then provide your answer."
        )

        answer = None

        # ── 1. Try Gemini ──────────────────────────────────────────────────
        if self.chat:
            try:
                response = self.chat.send_message(full_prompt)
                answer = response.text
            except Exception as e:
                print(f"  [Gemini error for {self.name}: {e}]")
                print("  → Falling back to OpenRouter / Nemotron …")

        # ── 2. Try OpenRouter fallback ────────────────────────────────────
        if answer is None and OPENROUTER_API_KEY:
            try:
                answer = _openrouter_chat(
                    system_prompt=self.system_prompt,
                    user_message=full_prompt,
                )
            except Exception as e:
                print(f"  [OpenRouter error for {self.name}: {e}]")

        # ── 3. Offline mode ───────────────────────────────────────────────
        if answer is None:
            answer = self._offline_response(question)

        self.conversation_history.append({"role": "user", "content": question})
        self.conversation_history.append({"role": "assistant", "content": answer})
        return answer

    def _offline_response(self, question: str) -> str:
        """Fallback response when all APIs are unavailable."""
        return (
            f"[{self.name} — Offline Mode]\n\n"
            f"Question received: {question}\n\n"
            "Neither Gemini nor OpenRouter API is available.\n"
            "Set GEMINI_API_KEY or OPENROUTER_API_KEY to enable AI analysis."
        )

    def clear_history(self):
        """Reset conversation history and start a fresh chat session."""
        self.conversation_history = []
        if client and self._config:
            self.chat = client.chats.create(model=self.model_name, config=self._config)

    def __repr__(self):
        backend = "Gemini" if client else ("OpenRouter/Nemotron" if OPENROUTER_API_KEY else "Offline")
        return f"<{self.name} Agent | backend={backend} | history={len(self.conversation_history)} msgs>"


print("✅ BaseAgent class defined")
print(f"   Active backend: {'Gemini' if client else ('OpenRouter/Nemotron' if OPENROUTER_API_KEY else 'Offline')}")


✅ BaseAgent class defined
   Active backend: Gemini


---

## 4. Persona 1: Process Engineer

A process engineer who can analyze the full ADU + VDU system, evaluate column performance, product yields, and recommend operational improvements.

In [52]:
PROCESS_ENGINEER_PROMPT = """You are a Senior Process Engineer specializing in crude oil refining,
specifically Atmospheric Distillation Units (ADU), Naphtha Stabilizer Units (NSU), and Vacuum Distillation Units (VDU).

**Your expertise includes:**
- Petroleum refining process design and optimization
- TBP (True Boiling Point) and ASTM distillation curve analysis
- Mass and energy balance across three-column distillation systems
- Product yield optimization across 13 product streams
- D95% distillation temperature specifications and quality control
- Heat exchanger network optimization and furnace efficiency
- Process simulation using DWSIM (Peng-Robinson EOS for petroleum systems)
- RL-based optimization with 16-dimensional action space

**System Configuration (3-column unit):**
- Feed: Crude oil at ~365°C, ~4736 kg/h

- **ADU (Atmospheric Distillation Unit)** — 5 products:
    1. Uncondensed_Gas   (fuel gas,   ~0.30 $/kg, overhead)
    2. Heavy_Naphtha     (reformer,   ~0.60 $/kg)   ← D95% ≤ 220°C
    3. SKO               (jet fuel,   ~0.75 $/kg)   ← D95% ≤ 300°C  [highest margin]
    4. Light_Gas_Oil     (lt diesel,  ~0.70 $/kg)   ← D95% ≤ 370°C
    5. Heavy_Gas_Oil     (hv diesel,  ~0.70 $/kg)   ← D95% ≤ 385°C
  Top pressure ~101 kPa; reboiler ~365°C; reflux ratio ~5.0

- **NSU (Naphtha Stabilizer Unit)** — 3 products:
    6. StabOffGas        (off-gas,    ~0.30 $/kg)
    7. LPG               (LPG,        ~0.65 $/kg)
    8. SRN               (naphtha,    ~0.75 $/kg)
  Pressurised column ~800 kPa; reboiler ~155°C

- **VDU (Vacuum Distillation Unit)** — 5 products:
    9.  Offgas           (gas,        ~0.30 $/kg)
    10. Vacuum_Diesel    (VDU diesel, ~0.70 $/kg)   ← D95% ≤ 385°C
    11. Vacuum_Gas_Oil   (FCC feed,   ~0.50 $/kg)   ← D95% ≤ 520°C
    12. Hotwell_Oil      (slop,       ~0.50 $/kg)
    13. Vac_residue      (bitumen,    ~0.35 $/kg)
  Deep vacuum ~8 kPa top; VDU furnace ~4200 kW

**RL Agent — 16-dim delta-action space:**
  ADU  (8): reflux_ratio, hn_draw_temp, sko_draw_temp, ld_draw_temp, hd_draw_temp,
            atmos_reboiler_temp, atmos_top_pressure, atmos_dp
  NSU  (2): nsu_reflux_ratio, nsu_reboiler_temp
  VDU  (6): vac_reflux_ratio, vac_reboiler_temp, vac_diesel_draw_temp, vgo_draw_temp,
            vac_top_pressure, vac_dp
  Actions are per-step deltas (±small value); progressive warmup 5%→100% over episode.

**Reward formula:**
  reward = [Σ(flow_i × price_i) − feed_cost − D95%_penalty − safety_penalty] / 100

**Analysis approach:**
1. Verify mass balance closure: Σ products ≈ feed (allow ~2% for simulation noise)
2. Check temperature profiles are monotonic in each column
3. Evaluate D95% quality compliance for Heavy_Naphtha, SKO, Light/Heavy_Gas_Oil, Vacuum_Diesel, Vacuum_Gas_Oil
4. Calculate economics: Revenue - Feed Cost - Utility Cost = Profit
5. Identify highest-margin products and constraints limiting their yield
6. Consider pressure/DP constraints — they affect separation sharpness

Be precise with numbers. Use proper engineering units. Reference actual data from context.
Format your analysis with clear sections and tables where appropriate."""


process_engineer = BaseAgent(
    name="Process Engineer",
    system_prompt=PROCESS_ENGINEER_PROMPT,
)
print(process_engineer)


<Process Engineer Agent | backend=Gemini | history=0 msgs>


In [53]:
# Example: Full system analysis
response = process_engineer.ask(
    "Analyze the current operating state of the ADU and VDU columns. "
    "Evaluate product yields, check mass balance closure, and identify "
    "any areas where the column performance could be improved.",
    context={
        "column_state": column_state,
        "product_prices": prices,
    }
)
print(response)

  [Gemini error for Process Engineer: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 38.328029743s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limit

In [54]:
# Example: Economic analysis
response = process_engineer.ask(
    "Calculate the hourly profit for the current operating conditions. "
    "Break down revenue by product stream, subtract feed cost and utility costs. "
    "Which product streams contribute the most to profitability? "
    "What operational changes would increase profit?",
    context={
        "column_state": column_state,
        "product_prices": prices,
    }
)
print(response)

  [Gemini error for Process Engineer: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 39.508844372s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limit

In [55]:
# Example: What-if analysis
response = process_engineer.ask(
    "What would happen if we increase the reflux ratio from 2.0 to 3.5? "
    "Analyze the impact on product purity, energy consumption, and overall profit. "
    "Also consider the effect on both ADU and VDU columns.",
    context={"column_state": column_state, "product_prices": prices}
)
print(response)

  [Gemini error for Process Engineer: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 42.125971802s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limit

---

## 5. Persona 2: Daily Report Developer

Generates structured daily operations reports covering all **13 product streams** across the ADU, NSU, and VDU columns,
with KPIs, D95% quality compliance, column pressure status, energy performance, economics, and RL agent status.


In [56]:
REPORT_DEVELOPER_PROMPT = """You are a Daily Operations Report Developer for a crude oil refinery's
three-column distillation system: ADU (Atmospheric Distillation Unit), NSU (Naphtha Stabilizer Unit),
and VDU (Vacuum Distillation Unit), producing 13 product streams in total.

**Product streams and value tiers:**
  HIGH VALUE  : SKO ($0.75), SRN ($0.75), Light_Gas_Oil ($0.70), Heavy_Gas_Oil ($0.70), Vacuum_Diesel ($0.70)
  MEDIUM VALUE: Heavy_Naphtha ($0.60), LPG ($0.65), Vacuum_Gas_Oil ($0.50), Hotwell_Oil ($0.50)
  LOWER VALUE : Uncondensed_Gas ($0.30), StabOffGas ($0.30), Offgas ($0.30), Vac_residue ($0.35)
  FEED COST   : Feed_Crude ($0.40/kg)

**D95% quality specifications (penalty 2.0 $/°C above limit):**
  Heavy_Naphtha ≤ 220°C | SKO ≤ 300°C | Light_Gas_Oil ≤ 370°C
  Heavy_Gas_Oil ≤ 385°C | Vacuum_Diesel ≤ 385°C | Vacuum_Gas_Oil ≤ 520°C

**Your role is to generate professional daily operations reports that include:**

1. **Executive Summary** — Key highlights, alerts, and overall plant status
2. **Production Summary** — Product yields for all 13 streams grouped by column (ADU / NSU / VDU),
   with comparison to typical targets and column mass balance check
3. **Energy Performance** — Furnace duties (atmospheric ~8500 kW, vacuum ~4200 kW),
   specific energy consumption (kW per kg of feed), condenser/reboiler duties
4. **Economic Summary** — Revenue breakdown by product stream, feed cost, net hourly profit,
   and profit contribution % per stream
5. **Quality Indicators** — D95% compliance for the 6 spec'd products; flag any exceedances
6. **Column Pressures** — ADU top pressure (~101 kPa), ADU DP (~15 kPa),
   VDU top pressure (~8 kPa), VDU DP (~7 kPa) — deviations indicate operational issues
7. **RL Agent Performance** — Training metrics, optimization recommendations,
   warmup stage (5%→100% over episode), solver tolerance status
8. **Recommendations** — Actionable items for the next shift

**Formatting requirements:**
- Use Markdown with headers, tables, and bullet points
- Include actual numbers from the data provided — never fabricate values
- Use engineering units: °C, kPa, kg/h, kW, $/hr
- Flag KPIs outside normal ranges with ⚠️
- Include a KPI dashboard table at the top

**Report structure:**
```
# Daily Operations Report — [Date]
## KPI Dashboard
| KPI | Value | Target | Status |
## 1. Executive Summary
## 2. Production Summary (ADU / NSU / VDU)
## 3. Energy Performance
## 4. Economic Summary
## 5. Quality & D95% Compliance
## 6. Column Pressures
## 7. RL Agent Status
## 8. Recommendations
```

Always be factual, concise, and actionable."""


report_developer = BaseAgent(
    name="Daily Report Developer",
    system_prompt=REPORT_DEVELOPER_PROMPT,
)
print(report_developer)


<Daily Report Developer Agent | backend=Gemini | history=0 msgs>


In [57]:
# Generate a daily report
today = datetime.now().strftime("%Y-%m-%d")

# Prepare training summary if available
training_summary = {}
if training_data:
    fm = training_data.get("final_metrics", {})
    training_summary = {
        "algorithm": training_data.get("config", {}).get("algorithm", "SAC"),
        "total_episodes": fm.get("episode", 0),
        "best_reward": fm.get("best_reward", 0),
        "avg_reward": fm.get("avg_reward", 0),
        "training_time_s": fm.get("training_time_seconds", 0),
    }

report = report_developer.ask(
    f"Generate the Daily Operations Report for {today}. "
    "Include all sections: KPI dashboard, production summary, energy performance, "
    "economic summary, quality indicators, safety status, and recommendations.",
    context={
        "date": today,
        "column_state": column_state,
        "product_prices": prices,
        "training_summary": training_summary,
    }
)
print(report)

  [Gemini error for Daily Report Developer: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 50.885200535s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate

In [58]:
# Save report to file
report_dir = PROJECT_ROOT / "Report" / "generated"
report_dir.mkdir(parents=True, exist_ok=True)

report_path = report_dir / f"daily_report_{today}.md"
with open(report_path, "w", encoding="utf-8") as f:
    f.write(f"# Daily Operations Report — {today}\n\n")
    f.write(report)

print(f"\n📄 Report saved to: {report_path}")


📄 Report saved to: d:\github\Distillation-column-agent\Report\generated\daily_report_2026-03-09.md


In [59]:
# Generate a shift-specific summary
response = report_developer.ask(
    "Generate a concise shift handover summary for the night shift. "
    "Focus on: critical parameters to watch, any pending alarms, "
    "and what the RL agent recommends for the next 8 hours.",
    context={"column_state": column_state, "product_prices": prices}
)
print(response)

  [Gemini error for Daily Report Developer: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 40.349224671s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate

---

## 6. Persona 3: Corrosion Expert

Specializes in overhead corrosion analysis in the ADU, focusing on:
- Overhead temperature monitoring
- Dew point corrosion (HCl, H₂S, NH₃)
- Salt deposition temperature
- Water condensation and acid formation
- Neutralization and filming amine recommendations

In [60]:
CORROSION_EXPERT_PROMPT = """You are a Corrosion & Materials Engineering Expert specializing in 
crude oil refinery overhead systems, particularly ADU (Atmospheric Distillation Unit) overhead corrosion.

**Your deep expertise covers:**

### Overhead Corrosion Mechanisms
1. **HCl Dew Point Corrosion** — Most critical. When steam + HCl condense, they form hydrochloric 
   acid at the dew point temperature, causing severe corrosion of carbon steel.
   - Key factor: overhead temperature relative to the water dew point
   - Critical when T_overhead approaches T_dew (typically 100-130°C at varying pressures)
   - Corrosion rate increases exponentially as pH drops below 4

2. **Ammonium Chloride (NH₄Cl) Salt Deposition**
   - Forms when NH₃ + HCl react above the water dew point
   - Salt deposition temperature depends on partial pressures of NH₃ and HCl
   - Deposits are hygroscopic → under-deposit corrosion when moisture is present
   - Typical salt point: 150-200°C depending on concentrations

3. **H₂S Corrosion** — Sulfidic corrosion in overhead system
   - Forms FeS scale which can be protective or non-protective
   - Interaction with HCl creates competitive corrosion

4. **Carbonic Acid (CO₂) Corrosion** — Minor but present from naphthenic acids decomposition

### Analysis Framework
For every analysis, evaluate:
1. **Water dew point** — Use Antoine equation: log₁₀(P) = A - B/(C+T)
   For water at overhead conditions: A=8.07131, B=1730.63, C=233.426 (mmHg, °C)
2. **NH₄Cl salt point** — Use the equilibrium: K_p = P_NH3 × P_HCl
   log₁₀(K_p) = 11.734 - 4364/T(K) (pressures in atm)
3. **Corrosion risk zones**:
   - GREEN: T_overhead > T_salt + 20°C (safe)
   - YELLOW: T_salt < T_overhead < T_salt + 20°C (caution)
   - RED: T_overhead < T_salt (active salt deposition)
   - CRITICAL: T_overhead ≤ T_dew (active acid corrosion)

### Mitigation Strategies
- Desalter optimization (reduce HCl precursors)
- Neutralizing amines (monoethanolamine, MDEA) — target pH 5.5-6.5
- Filming amines (imidazolines) — protective barrier
- Wash water injection rate optimization
- Metallurgy upgrades (Monel, titanium for overhead condensers)

### Output Format
Always provide:
- Calculated dew points and salt points with equations shown
- Risk assessment color-coded (GREEN/YELLOW/RED/CRITICAL)
- Specific chemical dosing recommendations with rates
- Inspection priorities and locations
- Trend analysis if historical data is available

Use proper units: °C, kPa, ppm, mg/L, mm/year (corrosion rates)."""


corrosion_expert = BaseAgent(
    name="Corrosion Expert",
    system_prompt=CORROSION_EXPERT_PROMPT,
)
print(corrosion_expert)

<Corrosion Expert Agent | backend=Gemini | history=0 msgs>


In [61]:
# Analyze current overhead corrosion risk
response = corrosion_expert.ask(
    "Analyze the current ADU overhead corrosion risk. "
    "Calculate the water dew point and NH₄Cl salt deposition temperature. "
    "Assess corrosion risk level and provide specific mitigation recommendations.",
    context={
        "overhead_conditions": {
            "overhead_temperature_C": column_state["overhead_temperature"],
            "overhead_pressure_kPa": column_state["overhead_pressure"],
            "water_content_mass_fraction": column_state["overhead_water_content"],
            "HCl_ppm": column_state["overhead_hcl_ppm"],
            "H2S_ppm": column_state["overhead_h2s_ppm"],
            "NH3_ppm": column_state["overhead_nh3_ppm"],
            "top_temperature_C": column_state["top_temperature"],
        },
        "column_conditions": {
            "feed_temperature_C": column_state["feed_temperature"],
            "bottom_temperature_C": column_state["bottom_temperature"],
            "condenser_duty_kW": column_state["condenser_duty"],
        },
    }
)
print(response)

  [Gemini error for Corrosion Expert: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 21.516750906s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limit

**Step‑by‑step analysis**

| Parameter | Value (given) | Unit |
|-----------|---------------|------|
| Overhead temperature | 50.0 | °C |
| Overhead pressure | 101.0 | kPa |
| Water mass‑fraction | 0.02 | – |
| HCl (v) | 5.0 | ppm |
| H₂S (v) | 15.0 | ppm |
| NH₃ (v) | 8.0 | ppm |
| Top (condenser) temperature | 50.0 | °C |

The analysis follows the corrosion‑risk framework you supplied.

---

## 1. Water‑dew‑point calculation  

The water partial pressure in the overhead vapour is  

\[
p_{\mathrm{H_2O}} = y_{\mathrm{H_2O}} \times P_{\text{total}}
\]

*Water mole fraction*  

\(y_{\mathrm{H_2O}} =\) water mass‑fraction (≈ mole‑fraction for dilute water) = **0.02**  

\[
p_{\mathrm{H_2O}} = 0.02 \times 101.0\;\text{kPa}= 2.02\;\text{kPa}
\]

Convert to **mm Hg** (the Antoine constants use mm Hg):

\[
1\;\text{mm Hg}=0.133322\;\text{kPa}\;\;\Rightarrow\;\;
p_{\mathrm{H_2O}}(\text{mm Hg})=\frac{2.02}{0.133322}=15.16\;\text{mm Hg}
\]

Antoine equation (water)  

\[
\log_{10} P = A - \frac

In [62]:
# What-if: overhead temperature changes
response = corrosion_expert.ask(
    "If the overhead temperature drops from 50°C to 35°C due to a weather event "
    "(cold front), what would be the impact on corrosion? "
    "Calculate the new dew point margin and recommend immediate actions.",
    context={
        "current_overhead_temp": 50.0,
        "new_overhead_temp": 35.0,
        "HCl_ppm": 5.0,
        "H2S_ppm": 15.0,
        "NH3_ppm": 8.0,
        "overhead_pressure_kPa": 101.0,
    }
)
print(response)

  [Gemini error for Corrosion Expert: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 28.736483196s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limit

In [63]:
# Corrosion monitoring plan
response = corrosion_expert.ask(
    "Develop a comprehensive corrosion monitoring plan for the ADU overhead system. "
    "Include: probe locations, monitoring frequency, KPIs and alarm limits, "
    "chemical treatment program (neutralizer and filming amine), "
    "and recommended inspection schedule.",
    context={"column_state": column_state}
)
print(response)

  [Gemini error for Corrosion Expert: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 30.054533535s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limit

---

## 7. Multi-Agent Orchestrator

Coordinate multiple agent personas to provide comprehensive analysis.

In [64]:
class AgentOrchestrator:
    """Orchestrates multiple AI agent personas for comprehensive analysis."""

    def __init__(self):
        self.agents = {
            "process_engineer": process_engineer,
            "report_developer": report_developer,
            "corrosion_expert": corrosion_expert,
        }

    def comprehensive_analysis(self, column_state: dict, prices: dict,
                                training_data: Optional[dict] = None) -> dict:
        """Run all agents on the current state and compile results."""
        results = {}
        context = {
            "column_state": column_state,
            "product_prices": prices,
        }
        if training_data:
            context["training_data"] = training_data

        # 1. Process Engineer: System overview
        print("🔧 Running Process Engineer analysis...")
        results["process_analysis"] = self.agents["process_engineer"].ask(
            "Provide a concise system performance analysis. "
            "Cover: mass balance, product yields, energy efficiency, and profit.",
            context=context,
        )

        # 2. Corrosion Expert: Overhead assessment
        print("🛡️ Running Corrosion Expert assessment...")
        corrosion_context = {
            "overhead_temperature_C": column_state.get("overhead_temperature", 50.0),
            "overhead_pressure_kPa": column_state.get("overhead_pressure", 101.0),
            "HCl_ppm": column_state.get("overhead_hcl_ppm", 5.0),
            "H2S_ppm": column_state.get("overhead_h2s_ppm", 15.0),
            "NH3_ppm": column_state.get("overhead_nh3_ppm", 8.0),
        }
        results["corrosion_assessment"] = self.agents["corrosion_expert"].ask(
            "Quick overhead corrosion risk assessment with risk level (GREEN/YELLOW/RED).",
            context={"overhead_conditions": corrosion_context},
        )

        # 3. Report Developer: Daily summary
        print("📋 Generating Daily Report...")
        results["daily_report"] = self.agents["report_developer"].ask(
            f"Generate the Daily Operations Report for {datetime.now().strftime('%Y-%m-%d')}.",
            context=context,
        )

        print("\n✅ All analyses complete!")
        return results

    def ask_agent(self, persona: str, question: str, context: Optional[dict] = None) -> str:
        """Route a question to a specific agent persona."""
        if persona not in self.agents:
            available = ", ".join(self.agents.keys())
            return f"Unknown persona '{persona}'. Available: {available}"
        return self.agents[persona].ask(question, context=context)


orchestrator = AgentOrchestrator()
print("✅ Agent Orchestrator initialized with", len(orchestrator.agents), "personas")

✅ Agent Orchestrator initialized with 3 personas


In [65]:
# Run comprehensive analysis across all agents
results = orchestrator.comprehensive_analysis(
    column_state=column_state,
    prices=prices,
    training_data=training_data,
)

print("\n" + "="*80)
print("PROCESS ENGINEER ANALYSIS")
print("="*80)
print(results["process_analysis"][:2000] + "..." if len(results["process_analysis"]) > 2000 else results["process_analysis"])

print("\n" + "="*80)
print("CORROSION ASSESSMENT")
print("="*80)
print(results["corrosion_assessment"][:2000] + "..." if len(results["corrosion_assessment"]) > 2000 else results["corrosion_assessment"])

🔧 Running Process Engineer analysis...
  [Gemini error for Process Engineer: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 19.972152447s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://a

---

## 8. Interactive Chat Interface

Chat with any agent persona interactively.

In [66]:
def chat_with_agent(persona: str = "process_engineer", question: str = ""):
    """Quick helper to chat with a specific agent."""
    context = {
        "column_state": column_state,
        "product_prices": prices,
    }
    response = orchestrator.ask_agent(persona, question, context=context)
    print(f"\n🤖 [{persona}]:\n")
    print(response)
    return response


# Example usage:
# chat_with_agent("process_engineer", "What is the current crude throughput efficiency?")
# chat_with_agent("corrosion_expert", "What is the safe minimum overhead temperature?")
# chat_with_agent("report_developer", "Generate an executive summary for management.")

In [67]:
# Try it out — ask the process engineer
chat_with_agent(
    "process_engineer",
    "What are the top 3 operational changes that would maximize profit right now?"
)

  [Gemini error for Process Engineer: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 32.482258982s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limit

'## Step‑by‑Step Profit Diagnosis  \n\n| Item | Current Value | Price ($/kg) | Revenue Contribution | Comments |\n|------|---------------|--------------|----------------------|----------|\n| **Uncondensed\u202fGas** | 25\u202fkg\u202fh⁻¹ | 0.30 | **7.5** | Low‑margin fuel gas – no realistic way to increase it without hurting higher‑margin streams. |\n| **Heavy\u202fNaphtha (HN)** | 106\u202fkg\u202fh⁻¹ | 0.68 (HN) | **72.1** | Mid‑margin; its cut is already tight (D95\u202f=\u202f155\u202f°C\u202f≤\u202f220\u202f°C). |\n| **SKO (Jet\u202fFuel)** | 43\u202fkg\u202fh⁻¹ | **0.75** | **32.3** | **Highest unit‑margin** product. Its D95 limit is 300\u202f°C, but the draw temperature is only 220\u202f°C – a lot of “room” to pull more heavy naphtha into SKO. |\n| **Light\u202fGas\u202fOil (LD)** | 51\u202fkg\u202fh⁻¹ | 0.60 | **30.6** | Margin is lower than SKO; sacrificing a few kg\u202fh⁻¹ of LD for extra SKO improves profit. |\n| **Heavy\u202fGas\u202fOil (HD)** | 69\u202fkg\u202fh⁻¹ | 0.55

In [68]:
# Ask the corrosion expert
chat_with_agent(
    "corrosion_expert",
    "Based on the current overhead conditions, should we increase the neutralizing amine injection rate?"
)

  [Gemini error for Corrosion Expert: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 13.95712041s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits

'### 1. \u202fDefine the operating point of the **overhead vapour**\n\n| Parameter | Value (from the JSON) | Unit |\n|-----------|-----------------------|------|\n| Overhead temperature, **Tₒᵥₑʰ** | **50\u202f°C** | °C |\n| Overhead pressure, **Pₒᵥₑʰ** | **101\u202fkPa** | kPa |\n| Water content (mass basis) | **0.02** | – (≈2\u202fwt\u202f% H₂O) |\n| HCl concentration | **5\u202fppm (v/v)** | – |\n| NH₃ concentration | **8\u202fppm (v/v)** | – |\n| H₂S concentration | **15\u202fppm (v/v)** | – |\n| Total pressure | 101\u202fkPa ≈ 1\u202fatm |\n\nThe key corrosion‑relevant equilibria are:\n\n1. **Water dew‑point (acid‑dew‑point) – when water condenses and aggressively attacks steel.**  \n2. **NH₄Cl salt‑point – the temperature at which ammonium chloride becomes thermodynamically stable and deposits.**  \n3. **Acid‑dew‑point for HCl (HCl+H₂O → HCl(aq)) – occurs when the water dew‑point is reached and HCl is present.**  \n\nWe will compute each of these temperatures from the data supplie

---

## 9. Save Full Report

Compile all agent outputs into a single comprehensive report file.

In [69]:
def save_comprehensive_report(results: dict, output_dir: Optional[Path] = None):
    """Save all agent analyses to a single Markdown report."""
    if output_dir is None:
        output_dir = PROJECT_ROOT / "Report" / "generated"
    output_dir.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    report_path = output_dir / f"comprehensive_report_{timestamp}.md"

    with open(report_path, "w", encoding="utf-8") as f:
        f.write("# Comprehensive CDU + NSU + VDU Analysis Report\n")
        f.write(f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        f.write("---\n\n")

        f.write("## Process Engineering Analysis\n\n")
        f.write(results.get("process_analysis", "N/A") + "\n\n")
        f.write("---\n\n")

        f.write("## Corrosion Risk Assessment\n\n")
        f.write(results.get("corrosion_assessment", "N/A") + "\n\n")
        f.write("---\n\n")

        f.write("## Daily Operations Report\n\n")
        f.write(results.get("daily_report", "N/A") + "\n\n")
        f.write("---\n\n")

        f.write("*Generated by CDU + NSU + VDU Optimizer AI Agent System*\n")

    print(f"📄 Comprehensive report saved to: {report_path}")
    return report_path


# Save the report
if 'results' in dir() and results:
    save_comprehensive_report(results)
else:
    print("Run the comprehensive analysis first (Section 7) to generate a report.")


📄 Comprehensive report saved to: d:\github\Distillation-column-agent\Report\generated\comprehensive_report_20260309_203249.md


---

## Summary

This notebook provides three AI agent personas:

| Persona | Usage |
|---------|-------|
| `process_engineer` | `chat_with_agent("process_engineer", "your question")` |
| `report_developer` | `chat_with_agent("report_developer", "your question")` |
| `corrosion_expert` | `chat_with_agent("corrosion_expert", "your question")` |

Use the **AgentOrchestrator** for comprehensive multi-agent analysis:
```python
results = orchestrator.comprehensive_analysis(column_state, prices, training_data)
```

All reports are saved to `Report/generated/`.

---
*More personas can be added by creating a new `BaseAgent` instance with a custom system prompt.*